In [1]:
!pip install -q transformers[torch] datasets evaluate accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 94.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cudf-cu12 26

In [2]:
import os
import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from sklearn.metrics import classification_report
import evaluate

base_dataset_path = "/kaggle/input/datasets/kghangco/vn-fb-news-dataset/"
kb_paths = {
    "kb1": os.path.join(base_dataset_path, "01_Raw_Data/01_Raw_Data"),
    "kb2": os.path.join(base_dataset_path, "02_Basic_Clean/02_Basic_Clean"),
    "kb3": os.path.join(base_dataset_path, "03_Full_Clean/03_Full_Clean"),
    "kb4": os.path.join(base_dataset_path, "04_No_Stopwords/04_No_Stopwords"),
    "kb5": os.path.join(base_dataset_path, "05_Balanced/05_Balanced")
}

# check gpu
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Thiết bị Kaggle đang cung cấp: {device.upper()}")
if device == "cpu":
    print(" -> Chọn T4 x2 hoặc P100)")

# dataset
class TextDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

    def __len__(self):
        return len(self.labels)

# metrics
metric_acc = evaluate.load("accuracy")
metric_f1 = evaluate.load("f1")
metric_precision = evaluate.load("precision")
metric_recall = evaluate.load("recall")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    preds = np.argmax(predictions, axis=1)
    return {
        "Accuracy": metric_acc.compute(predictions=preds, references=labels)["accuracy"],
        "Precision": metric_precision.compute(predictions=preds, references=labels, average="macro")["precision"],
        "Recall": metric_recall.compute(predictions=preds, references=labels, average="macro")["recall"],
        "F1-macro": metric_f1.compute(predictions=preds, references=labels, average="macro")["f1"]
    }

# tokenizer
model_name = "vinai/phobert-base"
print(f"Đang tải Tokenizer cho {model_name}...")
tokenizer = AutoTokenizer.from_pretrained(model_name)

# results
summary_results = []
print("Cấu hình Kaggle và hàm bổ trợ thành công!")

Thiết bị Kaggle đang cung cấp: CUDA


Đang tải Tokenizer cho vinai/phobert-base...


config.json:   0%|          | 0.00/557 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

bpe.codes: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Cấu hình Kaggle và hàm bổ trợ thành công!


In [3]:
import os
import pandas as pd

print("="*60)
print("TIEN HANH QUET CO CO LIET KE PATH VA CHECK COT")
print("="*60)

# Lay danh sach cac kich ban dang co tu bien kb_paths cua Phuc
for kb_name, folder_path in kb_paths.items():
    print(f"\n[KICH BAN: {kb_name.upper()}]")
    print(f"-> Folder path: {folder_path}")
    
    # Kiem tra nhanh file train cua tung kịch bản
    train_file_name = f"{kb_name}_train.csv"
    full_train_path = os.path.join(folder_path, train_file_name)
    
    if os.path.exists(full_train_path):
        print(f"Tim thay file: {train_file_name}")
        try:
            # Doc 2 dong dau de xem ten cot
            df_preview = pd.read_csv(full_train_path, nrows=2)
            cols = df_preview.columns.tolist()
            print(f"   - So luong cot: {len(cols)}")
            print(f"   - Danh sach ten cac cot dang co:")
            print(f"     {cols}")
        except Exception as e:
            print(f"Loi khi doc thu file: {e}")
    else:
        print(f"KHONG TIM THAY FILE: {train_file_name} tai duong dan tren!")
    print("-" * 60)

TIEN HANH QUET CO CO LIET KE PATH VA CHECK COT

[KICH BAN: KB1]
-> Folder path: /kaggle/input/datasets/kghangco/vn-fb-news-dataset/01_Raw_Data/01_Raw_Data
Tim thay file: kb1_train.csv
   - So luong cot: 92
   - Danh sach ten cac cot dang co:
     ['master_schema_version', 'record_id', 'crawler_owner', 'raw_schema_type', 'raw_file_path', 'raw_file_name', 'raw_record_index', 'post_id', 'post_url', 'content_hash', 'dedup_key', 'page_name', 'page_handle', 'page_url', 'page_followers', 'page_followers_raw', 'has_page_followers', 'post_type', 'external_url', 'post_content_for_labeling', 'post_content_length', 'word_count', 'topic_label_final', 'topic_label_id', 'label_status', 'label_source', 'annotator_1', 'annotator_2', 'annotation_round', 'annotation_note', 'is_uncertain_label', 'like_count', 'love_count', 'haha_count', 'wow_count', 'sad_count', 'angry_count', 'care_count', 'sorry_count', 'total_reactions', 'reaction_detail_sum', 'reaction_detail_coverage_ratio', 'has_reaction_total', 'ha

In [4]:
!rm -rf /kaggle/working/*

In [5]:
df_train = pd.read_csv("/kaggle/input/datasets/kghangco/vn-fb-news-dataset/01_Raw_Data/01_Raw_Data/kb1_train.csv")

In [6]:
import os
import gc
import torch
import numpy as np
import pandas as pd
import shutil  # Thêm thư viện để đóng gói zip
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer
from sklearn.metrics import classification_report

# 1. CO DINH BANG CHUYEN DOI NHAN TOAN CUC (Tranh loan ID giua cac kich ban)
all_unique_labels = sorted(["T01", "T02", "T03", "T04", "T05", "T06", "T07", "T08", "T09", "T10", "T11", "T12", "T13", "T14", "T15", "T16", "T17"])
label2id = {label: i for i, label in enumerate(all_unique_labels)}
id2label = {i: label for label, i in label2id.items()}
num_labels = len(all_unique_labels)

summary_results = []

for kb_name, folder_path in kb_paths.items():
    print("\n" + "="*70)
    print(f"HUAN LUYEN PHOBERT TREN KICH BAN: [{kb_name.upper()}]")
    print(f"Duong dan folder: {folder_path}")
    print("="*70)
    
    train_path = os.path.join(folder_path, f"{kb_name}_train.csv")
    val_path = os.path.join(folder_path, f"{kb_name}_val.csv")
    test_path = os.path.join(folder_path, f"{kb_name}_test.csv")
    
    if not os.path.exists(train_path):
        print(f"Khong tim thay file {kb_name}_train.csv tai folder. Bo qua.")
        continue
        
    df_train = pd.read_csv(train_path)
    df_val = pd.read_csv(val_path)
    df_test = pd.read_csv(test_path)
    
    # 2. DONG KIEM TRA VA CHON COT TEXT THUC TE CO TRONG FILE
    available_cols = df_train.columns.tolist()
    if kb_name == 'kb4' and 'text_nosw' in available_cols:
        text_col = 'text_nosw'
    elif kb_name in ['kb2', 'kb3', 'kb5'] and 'text_clean' in available_cols:
        text_col = 'text_clean'
    elif 'post_content_for_labeling' in available_cols:
        text_col = 'post_content_for_labeling'
    else:
        text_col = available_cols[19] # Vi tri mac dinh cua cot post_content_for_labeling neu co bien dong
        
    print(f"👉 QUYET DINH: Kich ban {kb_name.upper()} se hoc tren cot van ban: [{text_col}]")
    
    # 3. EP KIEU NHAN THEO BAN MAU CO DINH KHONG DOI
    y_train = df_train['topic_label_id'].map(label2id).astype(int).tolist()
    y_val = df_val['topic_label_id'].map(label2id).astype(int).tolist()
    y_test = df_test['topic_label_id'].map(label2id).astype(int).tolist()
    
    print("Dang tokenize van ban...")
    train_encodings = tokenizer(df_train[text_col].astype(str).tolist(), truncation=True, padding=True, max_length=256)
    val_encodings = tokenizer(df_val[text_col].astype(str).tolist(), truncation=True, padding=True, max_length=256)
    test_encodings = tokenizer(df_test[text_col].astype(str).tolist(), truncation=True, padding=True, max_length=256)
    
    train_dataset = TextDataset(train_encodings, y_train)
    val_dataset = TextDataset(val_encodings, y_val)
    test_dataset = TextDataset(test_encodings, y_test)
    
    model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=num_labels)
    
    # Đặt output thư mục train tạm thời
    tmp_output_dir = f"/kaggle/working/results_phobert_{kb_name}"
    
    training_args = TrainingArguments(
        output_dir=tmp_output_dir,
        num_train_epochs=10,              
        per_device_train_batch_size=16,   
        per_device_eval_batch_size=16,
        learning_rate=2e-5,               
        weight_decay=0.01,
        lr_scheduler_type="linear",       
        warmup_ratio=0.1,                  
        logging_dir=f"/kaggle/working/logs_{kb_name}",
        logging_steps=20,                  
        eval_strategy="epoch",           
        save_strategy="epoch",            
        save_total_limit=1,               
        load_best_model_at_end=True,   
        metric_for_best_model="F1-macro",
        greater_is_better=True,
        report_to="none"
    )
    
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        compute_metrics=compute_metrics
    )
    
    print(f"PhoBERT bat dau hoc kich ban [{kb_name.upper()}] (Toi da 10 Epochs)...")
    trainer.train()
    
    print(f"\nDANG DANH GIA MO HÌNH TREN TAP TEST CUA KICH BAN [{kb_name.upper()}]...")
    predictions = trainer.predict(test_dataset)
    y_pred = np.argmax(predictions.predictions, axis=1)
    
    target_names = [str(id2label[i]) for i in range(num_labels)]
    print(f"\nBANG DIEM CHI TIET TUNG LOP (PER-CLASS METRICS) CHO [{kb_name.upper()}]:")
    print(classification_report(y_test, y_pred, target_names=target_names, digits=4))
    
    test_metrics = compute_metrics((predictions.predictions, y_test))
    
    display_name = "KB5 (Oversampling)" if kb_name == "kb5" else kb_name.upper()
    
    metrics_summary = {
        "Model": "PhoBERT",
        "Data kịch bản": display_name,
        "Accuracy": round(test_metrics["Accuracy"], 4),
        "Precision": round(test_metrics["Precision"], 4),
        "Recall": round(test_metrics["Recall"], 4),
        "F1-macro": round(test_metrics["F1-macro"], 4)
    }
    summary_results.append(metrics_summary)

    # 💾 LƯU MODEL VÀ TOKENIZER CHUẨN BEST CHECKPOINT
    save_dir = f"/kaggle/working/phobert_{kb_name}_best_model"
    trainer.save_model(save_dir)       # Thay bằng trainer để lấy đúng bản lưu tốt nhất
    tokenizer.save_pretrained(save_dir)

    # 📦 NÉN ZIP VÀ XÓA THƯ MỤC THÔ ĐỂ TRÁNH ĐẦY Ổ ĐĨA KAGGLE
    shutil.make_archive(save_dir, 'zip', save_dir)
    if os.path.exists(save_dir):
        shutil.rmtree(save_dir)
    if os.path.exists(tmp_output_dir):
        shutil.rmtree(tmp_output_dir)
    print(f"📦 Đã nén thành công file: phobert_{kb_name}_best_model.zip")

    # 📝 LƯU CLASSIFICATION REPORT (.txt)
    report_str = classification_report(y_test, y_pred, target_names=target_names, digits=4)
    with open(f"/kaggle/working/phobert_{kb_name}_report.txt", "w", encoding="utf-8") as f:
        f.write(f"Kich ban: {kb_name.upper()}\n")
        f.write(report_str)

    # 📊 LƯU PREDICTIONS CSV (.csv)
    df_test_result = df_test.copy()
    df_test_result['y_true'] = [id2label[i] for i in y_test]
    df_test_result['y_pred'] = [id2label[i] for i in y_pred]
    df_test_result.to_csv(f"/kaggle/working/phobert_{kb_name}_predictions.csv", index=False)
    
    print(f"✅ Hoan thanh kich ban {kb_name.upper()}!\n")

    # 🧹 4. GIẢI PHÓNG BỘ NHỚ GPU CHUẨN (Chỉ chạy duy nhất 1 lần ngon lành)
    del model, trainer
    torch.cuda.empty_cache()
    gc.collect()



HUAN LUYEN PHOBERT TREN KICH BAN: [KB1]
Duong dan folder: /kaggle/input/datasets/kghangco/vn-fb-news-dataset/01_Raw_Data/01_Raw_Data
👉 QUYET DINH: Kich ban KB1 se hoc tren cot van ban: [post_content_for_labeling]
Dang tokenize van ban...


pytorch_model.bin:   0%|          | 0.00/543M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: vinai/phobert-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.bias              | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
roberta.pooler.dense.bias       | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
roberta.pooler.dense.weight     | UNEXPECTED | 
lm_head.decoder.bias            | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.decoder.weight          | UNEXPECTED | 
classifier.dense.weight         | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.out_proj.bias        | MISSING    | 
classifier.dense.bias           | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initia

model.safetensors:   0%|          | 0.00/543M [00:00<?, ?B/s]

...


DANG DANH GIA MO HÌNH TREN TAP TEST CUA KICH BAN [KB5]...


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]



BANG DIEM CHI TIET TUNG LOP (PER-CLASS METRICS) CHO [KB5]:
              precision    recall  f1-score   support

         T01     0.7059    0.8276    0.7619        58
         T02     0.6986    0.7727    0.7338        66
         T03     0.8158    0.8052    0.8105       154
         T04     0.7179    0.8485    0.7778        33
         T05     0.5909    0.7222    0.6500        18
         T06     0.5238    0.6471    0.5789        17
         T07     0.2857    0.2857    0.2857         7
         T08     0.8667    0.9286    0.8966        14
         T09     0.7568    0.7368    0.7467        38
         T10     0.8675    0.7443    0.8012       176
         T11     0.8621    0.9259    0.8929        27
         T12     0.2692    0.3333    0.2979        21
         T13     0.8025    0.6019    0.6878       108
         T14     0.6667    0.6667    0.6667         9
         T15     0.4091    0.5294    0.4615        17
         T16     0.8409    0.8222    0.8315        45
         T17     0.59

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

📦 Đã nén thành công file: phobert_kb5_best_model.zip
✅ Hoan thanh kich ban KB5!



In [7]:
print("\n" + "="*65)
print("BANG TONG HOP KET QUA DOI CHIEU TOAN DIEN (5 KICH BAN):")
print("="*65)
df_summary_report = pd.DataFrame(summary_results)
print(df_summary_report.to_markdown(index=False))

output_file_path = "/kaggle/working/phobert_general_metrics_report.csv"
df_summary_report.to_csv(output_file_path, index=False)

print("="*60)
print(f"🎉 File tổng hợp đã được lưu tại: {output_file_path}")
print("="*60)

# Trả về dataframe để Kaggle tự hiển thị bảng đẹp đẽ cuối ô code
df_summary_report


BANG TONG HOP KET QUA DOI CHIEU TOAN DIEN (5 KICH BAN):
| Model   | Data kịch bản      |   Accuracy |   Precision |   Recall |   F1-macro |
|:--------|:-------------------|-----------:|------------:|---------:|-----------:|
| PhoBERT | KB1                |     0.7792 |      0.7075 |   0.7095 |     0.7049 |
| PhoBERT | KB2                |     0.7792 |      0.7159 |   0.7168 |     0.7146 |
| PhoBERT | KB3                |     0.7828 |      0.7487 |   0.6898 |     0.7035 |
| PhoBERT | KB4                |     0.79   |      0.7259 |   0.7324 |     0.7262 |
| PhoBERT | KB5 (Oversampling) |     0.7434 |      0.6633 |   0.7077 |     0.6809 |
🎉 File tổng hợp đã được lưu tại: /kaggle/working/phobert_general_metrics_report.csv


,Model,Data kịch bản,Accuracy,Precision,Recall,F1-macro
0,PhoBERT,KB1,0.7792,0.7075,0.7095,0.7049
1,PhoBERT,KB2,0.7792,0.7159,0.7168,0.7146
2,PhoBERT,KB3,0.7828,0.7487,0.6898,0.7035
3,PhoBERT,KB4,0.7900,0.7259,0.7324,0.7262
4,PhoBERT,KB5 (Oversampling),0.7434,0.6633,0.7077,0.6809
